# Module 06 — CUDA Execution Model

This notebook explores kernels, threads, blocks, grids, scheduling, synchronization, grid-stride loops, and launch overhead on a CUDA-enabled GPU.

Run it on the Lightning NVIDIA T4 used by the course.

In [ ]:
# This notebook implements the CUDA kernels directly; it does not import the companion script.
import statistics
import time

import numpy as np
import torch
from numba import cuda, float32

assert cuda.is_available() and torch.cuda.is_available(), 'Select a CUDA-enabled T4 runtime.'

@cuda.jit
def inspect_threads(output):
    # Write block, local-thread, global, and warp indices for inspection.
    global_id = cuda.grid(1)
    if global_id < output.shape[0]:
        output[global_id, 0] = cuda.blockIdx.x
        output[global_id, 1] = cuda.threadIdx.x
        output[global_id, 2] = global_id
        output[global_id, 3] = cuda.threadIdx.x // 32

@cuda.jit
def grid_stride_double(source, destination, size):
    # The stride lets launched threads cover more elements than one pass.
    index = cuda.grid(1)
    stride = cuda.gridsize(1)
    for i in range(index, size, stride):
        destination[i] = 2.0 * source[i]

@cuda.jit
def reduce_block(source, output):
    # Shared memory and a barrier implement a tree reduction.
    tid = cuda.threadIdx.x
    shared = cuda.shared.array(shape=256, dtype=float32)
    shared[tid] = source[tid]
    cuda.syncthreads()
    stride = 128
    while stride > 0:
        if tid < stride:
            shared[tid] += shared[tid + stride]
        cuda.syncthreads()
        stride //= 2
    if tid == 0:
        output[0] = shared[0]

def inspect_gpu():
    device = cuda.get_current_device()
    props = torch.cuda.get_device_properties(0)
    print('GPU:', device.name)
    print('Compute capability:', device.compute_capability)
    print('SMs:', device.MULTIPROCESSOR_COUNT)
    print('Max threads/block:', device.MAX_THREADS_PER_BLOCK)
    print('Threads/SM:', getattr(props, 'max_threads_per_multi_processor', 'unknown'))

def run_thread_inspection(blocks=3, threads=64):
    output = cuda.device_array((blocks * threads, 4), dtype=np.int32)
    inspect_threads[blocks, threads](output)
    cuda.synchronize()
    result = output.copy_to_host()
    print(result[:10])
    for index in sorted({0, 31, 32, blocks * threads - 1}):
        if index < result.shape[0]:
            print('global=', index, 'block=', result[index, 0], 'local=', result[index, 1], 'warp=', result[index, 3])

def run_grid_stride(size=1_000_000, blocks=80, threads=256):
    source = np.arange(size, dtype=np.float32)
    device_source = cuda.to_device(source)
    device_destination = cuda.device_array_like(device_source)
    grid_stride_double[blocks, threads](device_source, device_destination, size)
    cuda.synchronize()
    result = device_destination.copy_to_host()
    print('threads launched:', blocks * threads, 'correct:', np.allclose(result, source * 2))

def run_block_reduction():
    source = cuda.to_device(np.ones(256, dtype=np.float32))
    output = cuda.device_array(1, dtype=np.float32)
    reduce_block[1, 256](source, output)
    cuda.synchronize()
    print('reduction:', output.copy_to_host()[0], 'expected: 256.0')

def run_launch_overhead(repeats=100, trials=20):
    x = torch.ones(1024, device='cuda')
    y = torch.empty_like(x)
    batched_x = torch.ones((repeats, 1024), device='cuda')
    batched_y = torch.empty_like(batched_x)
    def many():
        for _ in range(repeats):
            torch.mul(x, 2.0, out=y)
    def one():
        torch.mul(batched_x, 2.0, out=batched_y)
    for _ in range(5): many(); one()
    def measure(fn):
        samples = []
        for _ in range(trials):
            torch.cuda.synchronize(); start = time.perf_counter(); fn(); torch.cuda.synchronize()
            samples.append((time.perf_counter() - start) * 1000)
        return statistics.median(samples)
    many_ms, one_ms = measure(many), measure(one)
    print('many small launches:', many_ms, 'ms; one launch:', one_ms, 'ms; ratio:', many_ms / one_ms)

inspect_gpu()

## Experiment A — inspect thread and block IDs

This launch uses three blocks with 64 threads per block. Inspect the boundary values where warps and blocks change.

In [ ]:
# Experiment A: launch three blocks and inspect the actual index mapping.
blocks, threads = 3, 64
output = cuda.device_array((blocks * threads, 4), dtype=np.int32)
inspect_threads[blocks, threads](output)
cuda.synchronize()
thread_table = output.copy_to_host()
print('first rows:', thread_table[:10])
for index in [0, 31, 32, 63, 64, blocks * threads - 1]:
    print('global=', index, 'block=', thread_table[index, 0], 'local=', thread_table[index, 1], 'warp=', thread_table[index, 3])

## Experiment B — grid-stride loop

Only 20,480 threads are launched, but they process one million elements by repeatedly advancing by the total grid size.

In [ ]:
# Experiment B: one grid-stride kernel processes many more elements than threads.
size, blocks, threads = 1_000_000, 80, 256
host_source = np.arange(size, dtype=np.float32)
source = cuda.to_device(host_source)
destination = cuda.device_array_like(source)
grid_stride_double[blocks, threads](source, destination, size)
cuda.synchronize()
host_result = destination.copy_to_host()
print('threads launched:', blocks * threads)
print('correct:', np.allclose(host_result, host_source * 2))

## Experiment C — shared-memory block reduction

One block reduces 256 values. Each stage uses block-wide synchronization before the next stage reads the partial sums.

In [ ]:
# Experiment C: reduce 256 values in shared memory with a barrier per stage.
host_values = np.ones(256, dtype=np.float32)
source = cuda.to_device(host_values)
result = cuda.device_array(1, dtype=np.float32)
reduce_block[1, 256](source, result)
cuda.synchronize()
gpu_sum = result.copy_to_host()[0]
print('GPU sum:', gpu_sum, 'expected:', host_values.sum())
assert np.isclose(gpu_sum, host_values.sum())

## Experiment D — launch overhead

Compare 100 separate elementwise launches with one larger launch that performs the same broad class of work. The measurement includes host synchronization and is intended to illustrate dispatch overhead.

In [ ]:
# Experiment D: compare many small dispatches with one batched dispatch.
repeats, trials = 100, 20
x = torch.ones(1024, device='cuda'); y = torch.empty_like(x)
batched_x = torch.ones((repeats, 1024), device='cuda'); batched_y = torch.empty_like(batched_x)
def many_launches():
    for _ in range(repeats): torch.mul(x, 2.0, out=y)
def one_launch(): torch.mul(batched_x, 2.0, out=batched_y)
for _ in range(5): many_launches(); one_launch()
def wall_ms(fn):
    samples = []
    for _ in range(trials):
        torch.cuda.synchronize(); start = time.perf_counter(); fn(); torch.cuda.synchronize()
        samples.append((time.perf_counter() - start) * 1000)
    return statistics.median(samples)
many_ms, one_ms = wall_ms(many_launches), wall_ms(one_launch)
print('many launches:', many_ms, 'ms; one launch:', one_ms, 'ms; ratio:', many_ms / one_ms)

## Record your results

Record the GPU name, compute capability, Python environment, launch dimensions, correctness checks, timing samples, and the interpretation of each experiment.

Key questions:

1. Where do local thread IDs restart?
2. Why can a grid-stride loop process more elements than launched threads?
3. Why must all threads reach the shared-memory barrier?
4. Why can one large launch be faster than many small launches?